In [0]:
%run "./00_Pipeline_Configurations"


In [0]:
%run "./08_Audit_Log_Function"

In [0]:
# 05_Silver_Layer_Attendance

from datetime import datetime

start_time = datetime.now()

try:
    df_attendance = spark.table(bronze_attendance_table)
    rows_before = df_attendance.count()

    df_attendance_clean = (
        df_attendance
        .dropDuplicates(["student_id"])
        .dropna()   # null attendance_pct, math_marks ya science_marks wali row hata do
    )

    rows_after = df_attendance_clean.count()

    df_attendance_clean.write.format("delta").mode("overwrite").saveAsTable(silver_attendance_table)

    display(df_attendance_clean)

    print(f"Rows before cleaning: {rows_before}")
    print(f"Rows after cleaning: {rows_after}")
    print(f"Rows dropped (nulls/duplicates): {rows_before - rows_after}")

    log_audit("silver_attendance", table_name=silver_attendance_table, status="SUCCESS", start_time=start_time)

except Exception as e:
    log_audit("silver_attendance", status="FAILED", start_time=start_time, error_msg=str(e))
    raise